# Experiments

Prototype notebook for data processing experiments.

## Experiment 1: frontal, no-baseline RSA inputs

This experiment generates four trial-level EEG datasets for each participant:

- `frp`: average 600 ms post-fixation-onset activity across valid fixations on sequence elements.
- `frp_control`: trial-matched random 600 ms windows, with the number of windows matched to the number of valid sequence fixations in that trial and sampled onsets separated by at least 300 ms. By default, these are sampled from `stim-all_stim` to `trial_end`.
- `response`: 600 ms pre-response activity, locked to valid response events (`a`, `x`, `m`, `l`).
- `rest`: 600 ms pre-trial-start activity, locked to `trial_start`.

Global constraints for this notebook:

- Only frontal electrodes are used.
- No baseline correction is applied anywhere.
- Outputs are written into a timestamped run folder under a user-specified base folder.

In [13]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime

import json
import os
import sys
import pickle


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if path.name == "abstract_reasoning" and (path / "analysis").exists():
            return path
    raise FileNotFoundError("Could not locate the abstract_reasoning repo root.")


ROOT = find_repo_root()
ANALYSIS_DIR = ROOT / "analysis"
SRC_DIR = ANALYSIS_DIR / "src"

for path in [SRC_DIR, ANALYSIS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".temp/matplotlib"))

ROOT

PosixPath('/Users/chris/Documents/PhD-Local/abstract_reasoning')

In [14]:
import numpy as np
import pandas as pd
import mne
from tqdm.auto import tqdm

from rsatoolbox.data import Dataset
from rsatoolbox.rdm import calc_rdm, RDMs

from ar_analysis.analysis_config import Config as c
from ar_analysis.data_loader.human import HumanGroupData, HumanSubjData, HumanSessData
from ar_analysis.utils.analysis_utils import save_pickle

mne.set_log_level("WARNING")

## Experiment 1 Configuration

Set `OUTPUT_BASE_DIR` before running. The notebook will create a timestamped run folder inside it. The default below writes to `.temp/experiments`, which is safe for prototyping but should be changed for full runs.

In [15]:
# -----------------------------
# User-editable configuration
# -----------------------------
DATA_ROOT = Path("/Volumes/Realtek 1Tb/PhD Data/experiment1/data")
HUMAN_DATA_DIR = DATA_ROOT / "Lab/raw-BIDS3"
PREPROCESSED_DIR = DATA_ROOT / "Lab/preprocessed"

# Change this for full runs, e.g. DATA_ROOT / "Lab/analyzed/experiments".
OUTPUT_BASE_DIR = ROOT / ".temp/experiments"

# Use None to process all discovered participants. Use a list for prototyping, e.g. [1].
SUBJ_NS: list[int] | None = [2]

DATA_FMT = "bids"
RNG_SEED = 0

# Global experiment constraints.
FRONTAL_CHANS = list(c.EEG_CHAN_GROUPS.frontal)
NO_BASELINE = None

# All windows are 600 ms. FRP/control are post-onset; response/rest are pre-event.
WINDOW_S = 0.600
FRP_TMIN, FRP_TMAX = 0.0, WINDOW_S
RESPONSE_TMIN, RESPONSE_TMAX = -WINDOW_S, 0.0
REST_TMIN, REST_TMAX = -WINDOW_S, 0.0

RESPONSE_EVENTS = ["a", "x", "m", "l"]
REST_EVENT = "trial_start"
FRP_STIM_SCOPE = "sequence"

# Random-control windows are sampled within each trial's true trial interval.
RANDOM_CONTROL_START_EVENT = "stim-all_stim"
RANDOM_CONTROL_END_EVENT = "trial_end"
RANDOM_CONTROL_MIN_ONSET_DIFF_S = 0.300

# Dataset/RDM export settings.
DISSIMILARITY_METRIC = "correlation"
DATASET_LEVELS = ["trial_lvl", "pattern_lvl"]
EXPORT_EVOKED_OBJECTS = False

print(f"HUMAN_DATA_DIR:   {HUMAN_DATA_DIR} exists={HUMAN_DATA_DIR.exists()}")
print(f"PREPROCESSED_DIR: {PREPROCESSED_DIR} exists={PREPROCESSED_DIR.exists()}")
print(f"OUTPUT_BASE_DIR:  {OUTPUT_BASE_DIR}")
print(f"Frontal channels ({len(FRONTAL_CHANS)}): {FRONTAL_CHANS}")

HUMAN_DATA_DIR:   /Volumes/Realtek 1Tb/PhD Data/experiment1/data/Lab/raw-BIDS3 exists=False
PREPROCESSED_DIR: /Volumes/Realtek 1Tb/PhD Data/experiment1/data/Lab/preprocessed exists=False
OUTPUT_BASE_DIR:  /Users/chris/Documents/PhD-Local/abstract_reasoning/.temp/experiments
Frontal channels (17): ['Fp1', 'Fpz', 'Fp2', 'AF7', 'AF3', 'AFz', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8']


In [16]:
def create_run_dirs(base_dir: Path, experiment_name: str = "experiment1") -> dict[str, Path]:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = base_dir / f"{experiment_name}_{timestamp}"
    dirs = {
        "run": run_dir,
        "trial_data": run_dir / "trial_data",
        "datasets": run_dir / "rsatoolbox_datasets",
        "rdms": run_dir / "rdms",
        "logs": run_dir / "logs",
        "metadata": run_dir / "metadata",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=False if path == run_dir else True)
    return dirs


RUN_DIRS = create_run_dirs(OUTPUT_BASE_DIR, "experiment1_frontal_no_baseline")

CONFIG = {
    "data_root": str(DATA_ROOT),
    "human_data_dir": str(HUMAN_DATA_DIR),
    "preprocessed_dir": str(PREPROCESSED_DIR),
    "output_base_dir": str(OUTPUT_BASE_DIR),
    "run_dir": str(RUN_DIRS["run"]),
    "subj_Ns": SUBJ_NS,
    "data_fmt": DATA_FMT,
    "rng_seed": RNG_SEED,
    "channels": FRONTAL_CHANS,
    "baseline": None,
    "window_s": WINDOW_S,
    "frp_stim_scope": FRP_STIM_SCOPE,
    "response_events": RESPONSE_EVENTS,
    "rest_event": REST_EVENT,
    "random_control_start_event": RANDOM_CONTROL_START_EVENT,
    "random_control_end_event": RANDOM_CONTROL_END_EVENT,
    "random_control_min_onset_diff_s": RANDOM_CONTROL_MIN_ONSET_DIFF_S,
    "dissimilarity_metric": DISSIMILARITY_METRIC,
    "dataset_levels": DATASET_LEVELS,
    "export_evoked_objects": EXPORT_EVOKED_OBJECTS,
}

with open(RUN_DIRS["metadata"] / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

RUN_DIRS

{'run': PosixPath('/Users/chris/Documents/PhD-Local/abstract_reasoning/.temp/experiments/experiment1_frontal_no_baseline_20260605_205748'),
 'trial_data': PosixPath('/Users/chris/Documents/PhD-Local/abstract_reasoning/.temp/experiments/experiment1_frontal_no_baseline_20260605_205748/trial_data'),
 'datasets': PosixPath('/Users/chris/Documents/PhD-Local/abstract_reasoning/.temp/experiments/experiment1_frontal_no_baseline_20260605_205748/rsatoolbox_datasets'),
 'rdms': PosixPath('/Users/chris/Documents/PhD-Local/abstract_reasoning/.temp/experiments/experiment1_frontal_no_baseline_20260605_205748/rdms'),
 'logs': PosixPath('/Users/chris/Documents/PhD-Local/abstract_reasoning/.temp/experiments/experiment1_frontal_no_baseline_20260605_205748/logs'),
 'metadata': PosixPath('/Users/chris/Documents/PhD-Local/abstract_reasoning/.temp/experiments/experiment1_frontal_no_baseline_20260605_205748/metadata')}

## Helper Functions

These helpers keep the four conditions aligned around the same representation: one `mne.Evoked` per trial, each containing frontal channels only and no baseline correction.

In [17]:
def trial_events(raw_trial: mne.io.Raw) -> pd.DataFrame:
    """Return annotation-derived events with sample indices relative to raw_trial data."""
    events, event_id = mne.events_from_annotations(raw_trial, verbose="WARNING")
    inv_event_id = {v: k for k, v in event_id.items()}
    rows = []
    for sample_abs, _, code in events:
        rows.append(
            {
                "sample": int(sample_abs - raw_trial.first_samp),
                "description": inv_event_id[int(code)],
            }
        )
    return pd.DataFrame(rows)


def n_samples_for_window(raw_trial: mne.io.Raw, tmin: float, tmax: float) -> int:
    return int(np.ceil((tmax - tmin) * raw_trial.info["sfreq"])) + 1


def selected_info(raw_trial: mne.io.Raw, selected_chans: list[str]) -> mne.Info:
    return raw_trial.copy().pick(selected_chans).info


def make_evoked_from_array(
    data: np.ndarray,
    info: mne.Info,
    tmin: float,
    comment: str,
) -> mne.Evoked:
    return mne.EvokedArray(data, info, tmin=tmin, nave=1, comment=comment)


def extract_event_locked_window(
    raw_trial: mne.io.Raw,
    event_names: list[str],
    tmin: float,
    tmax: float,
    selected_chans: list[str],
    comment: str,
) -> mne.Evoked | None:
    """Extract one no-baseline event-locked window from a trial."""
    events = trial_events(raw_trial)
    if events.empty:
        return None

    event_rows = events.query("description in @event_names")
    if event_rows.empty:
        return None

    event_sample = int(event_rows.iloc[0]["sample"])
    sfreq = raw_trial.info["sfreq"]
    n_samples = n_samples_for_window(raw_trial, tmin, tmax)
    start_sample = event_sample + int(np.round(tmin * sfreq))
    stop_sample = start_sample + n_samples

    data = raw_trial.get_data(picks=selected_chans)
    if start_sample < 0 or stop_sample > data.shape[1]:
        return None

    return make_evoked_from_array(
        data[:, start_sample:stop_sample],
        selected_info(raw_trial, selected_chans),
        tmin=tmin,
        comment=comment,
    )


def extract_response_epoch(raw_trial: mne.io.Raw) -> mne.Evoked | None:
    return extract_event_locked_window(
        raw_trial=raw_trial,
        event_names=RESPONSE_EVENTS,
        tmin=RESPONSE_TMIN,
        tmax=RESPONSE_TMAX,
        selected_chans=FRONTAL_CHANS,
        comment="response_pre600_no_baseline",
    )


def extract_rest_epoch(raw_trial: mne.io.Raw) -> mne.Evoked | None:
    return extract_event_locked_window(
        raw_trial=raw_trial,
        event_names=[REST_EVENT],
        tmin=REST_TMIN,
        tmax=REST_TMAX,
        selected_chans=FRONTAL_CHANS,
        comment="rest_pre_trial_start_600_no_baseline",
    )

In [18]:
def trial_period_samples(
    raw_trial: mne.io.Raw,
    start_event: str = RANDOM_CONTROL_START_EVENT,
    end_event: str = RANDOM_CONTROL_END_EVENT,
) -> tuple[int, int] | None:
    """Return sample bounds for the true trial interval inside a cropped trial Raw."""
    events = trial_events(raw_trial)
    if events.empty:
        return None

    start_rows = events.query("description == @start_event")
    end_rows = events.query("description == @end_event")
    if start_rows.empty or end_rows.empty:
        return None

    start_sample = int(start_rows.iloc[0]["sample"])
    end_sample = int(end_rows.iloc[-1]["sample"])
    if end_sample <= start_sample:
        return None
    return start_sample, end_sample


def sample_spaced_random_starts(
    possible_starts: np.ndarray,
    n_windows: int,
    min_onset_diff_samples: int,
    rng: np.random.Generator,
    max_attempts: int = 1_000,
) -> np.ndarray | None:
    """Sample unique onset samples separated by at least min_onset_diff_samples."""
    if n_windows <= 0:
        return np.asarray([], dtype=int)
    if n_windows > possible_starts.size:
        return None

    min_onset_diff_samples = max(1, int(min_onset_diff_samples))
    for _ in range(max_attempts):
        selected: list[int] = []
        for start in rng.permutation(possible_starts):
            if all(abs(int(start) - other) >= min_onset_diff_samples for other in selected):
                selected.append(int(start))
                if len(selected) == n_windows:
                    return np.sort(np.asarray(selected, dtype=int))
    return None


def extract_random_control_frp(
    raw_trial: mne.io.Raw,
    n_windows: int,
    rng: np.random.Generator,
    selected_chans: list[str] = FRONTAL_CHANS,
    window_s: float = WINDOW_S,
    min_onset_diff_s: float = RANDOM_CONTROL_MIN_ONSET_DIFF_S,
) -> tuple[mne.Evoked | None, list[int]]:
    """Generate an average of n random no-baseline windows in a trial."""
    if n_windows <= 0:
        return None, []

    period = trial_period_samples(raw_trial)
    if period is None:
        return None, []

    start_bound, end_bound = period
    sfreq = raw_trial.info["sfreq"]
    n_samples = int(np.ceil(window_s * sfreq)) + 1
    max_start = end_bound - n_samples
    if max_start < start_bound:
        return None, []

    possible_starts = np.arange(start_bound, max_start + 1, dtype=int)
    min_onset_diff_samples = int(np.ceil(min_onset_diff_s * sfreq))
    starts = sample_spaced_random_starts(
        possible_starts=possible_starts,
        n_windows=n_windows,
        min_onset_diff_samples=min_onset_diff_samples,
        rng=rng,
    )
    if starts is None:
        return None, []

    data = raw_trial.get_data(picks=selected_chans)
    windows = np.stack([data[:, start : start + n_samples] for start in starts])
    avg_window = windows.mean(axis=0)

    evoked = make_evoked_from_array(
        avg_window,
        selected_info(raw_trial, selected_chans),
        tmin=0.0,
        comment="frp_random_control_post600_no_baseline",
    )
    return evoked, starts.tolist()


def extract_true_frp(
    sess: HumanSessData,
    eeg_trial: mne.io.Raw,
    et_trial: mne.io.Raw,
    behav: pd.DataFrame,
    trial_N: int,
) -> tuple[mne.Evoked | None, int]:
    """Extract the true sequence-FRP and return the number of valid fixation epochs."""
    frp, fixation_epochs = sess.get_trial_frp(
        eeg_trial=eeg_trial,
        et_trial=et_trial,
        raw_behav=behav,
        trial_N=trial_N,
        stim_scope=FRP_STIM_SCOPE,
        tmin=FRP_TMIN,
        tmax=FRP_TMAX,
        baseline=NO_BASELINE,
        selected_chans=FRONTAL_CHANS,
        return_epochs=True,
    )
    n_fixations = 0 if fixation_epochs is None else len(fixation_epochs)
    return frp, n_fixations

## Subject Processing

Each condition is first generated independently. Then trials are intersected across all four conditions so the RSA inputs are perfectly matched.

In [19]:
CONDITIONS = ["frp", "frp_control", "response", "rest"]


def trial_uid(subj_N: int, sess_N: int, trial_N: int) -> str:
    return f"sub-{subj_N:02}_ses-{sess_N:02}_trial-{trial_N:03}"


def row_value(row: pd.Series, key: str, default=None):
    return row[key] if key in row.index else default


def process_subject_trials(
    subj: HumanSubjData,
    rng: np.random.Generator,
) -> tuple[dict[str, dict[str, mne.Evoked]], dict[str, dict], list[dict]]:
    condition_data: dict[str, dict[str, mne.Evoked]] = {condition: {} for condition in CONDITIONS}
    trial_records: dict[str, dict] = {}
    random_window_rows: list[dict] = []

    for sess_N, sess in tqdm(
        subj.sessions.items(),
        desc=f"subj {subj.subj_N:02} sessions",
        leave=False,
    ):
        behav, et_trials, eeg_trials = sess.get_trials_data(
            preprocessed_dir=PREPROCESSED_DIR,
            raise_error=False,
            eeg_incomplete="skip",
        )
        if behav is None or et_trials is None or eeg_trials is None:
            continue

        et_trials = list(et_trials)
        eeg_trials = list(eeg_trials)
        n_trials = min(len(behav), len(et_trials), len(eeg_trials))

        for trial_pos, ((behav_index, behav_row), et_trial, eeg_trial) in enumerate(
            zip(behav.iterrows(), et_trials, eeg_trials)
        ):
            if trial_pos >= n_trials:
                break

            # get_trial_info expects the behavioral DataFrame index used by the session loader.
            trial_N = int(behav_index)
            uid = trial_uid(subj.subj_N, sess_N, trial_N)

            record = {
                "trial_uid": uid,
                "subj_N": subj.subj_N,
                "sess_N": sess_N,
                "trial_N": trial_N,
                "trial_pos": trial_pos,
                "item_id": row_value(behav_row, "item_id"),
                "pattern": row_value(behav_row, "pattern"),
            }
            trial_records[uid] = record

            frp, n_fixations = extract_true_frp(sess, eeg_trial, et_trial, behav, trial_N)
            if frp is not None:
                condition_data["frp"][uid] = frp

            control, random_starts = extract_random_control_frp(eeg_trial, n_fixations, rng)
            if control is not None:
                condition_data["frp_control"][uid] = control
            for start in random_starts:
                random_window_rows.append(
                    {
                        "trial_uid": uid,
                        "subj_N": subj.subj_N,
                        "sess_N": sess_N,
                        "trial_N": trial_N,
                        "random_onset_sample": start,
                        "random_onset_s": start / eeg_trial.info["sfreq"],
                        "min_onset_diff_s": RANDOM_CONTROL_MIN_ONSET_DIFF_S,
                    }
                )

            response = extract_response_epoch(eeg_trial)
            if response is not None:
                condition_data["response"][uid] = response

            rest = extract_rest_epoch(eeg_trial)
            if rest is not None:
                condition_data["rest"][uid] = rest

    return condition_data, trial_records, random_window_rows


def dataset_trial_uids(trial_records: dict[str, dict]) -> list[str]:
    """Return all trials observed for a participant; missing condition data is filled with NaNs later."""
    return sorted(trial_records)


def complete_trial_uids(condition_data: dict[str, dict[str, mne.Evoked]]) -> list[str]:
    """Return trials with valid data in every condition, useful for summaries/checks."""
    trial_sets = [set(condition_data[condition].keys()) for condition in CONDITIONS]
    return sorted(set.intersection(*trial_sets))


In [20]:

def evoked_to_feature_vector(evoked: mne.Evoked) -> np.ndarray:
    """Flatten frontal channels x time into one observation feature vector."""
    return evoked.get_data().reshape(-1)


def sorted_aligned_uids(aligned_uids: list[str], trial_records: dict[str, dict]) -> list[str]:
    """Sort aligned trials by pattern first and item_id second."""
    manifest = pd.DataFrame([trial_records[uid] for uid in aligned_uids]).copy()
    manifest["_uid"] = aligned_uids
    manifest = manifest.sort_values(
        ["pattern", "item_id", "sess_N", "trial_N"],
        kind="mergesort",
        na_position="last",
    )
    return manifest["_uid"].tolist()


def infer_feature_shape(condition_trials: dict[str, dict[str, mne.Evoked]]) -> tuple[int, ...]:
    """Find a valid trial feature shape for NaN-filled missing observations."""
    for trials in condition_trials.values():
        for evoked in trials.values():
            return evoked_to_feature_vector(evoked).shape
    raise ValueError("Could not infer feature shape: no valid condition data found.")


def dataset_descriptors(subj_N: int, condition: str, level: str) -> dict:
    return {
        "subj_N": subj_N,
        "condition": condition,
        "level": level,
        "channels": "frontal",
        "baseline": "None",
        "window_s": WINDOW_S,
        "missing_observations": "np.nan",
    }


def build_trial_dataset(
    subj_N: int,
    condition: str,
    condition_trials: dict[str, mne.Evoked],
    trial_records: dict[str, dict],
    aligned_uids: list[str],
    feature_shape: tuple[int, ...],
) -> tuple[Dataset, np.ndarray, pd.DataFrame]:
    sorted_uids = sorted_aligned_uids(aligned_uids, trial_records)
    measurement_rows = []
    has_data = []
    for uid in sorted_uids:
        evoked = condition_trials.get(uid)
        if evoked is None:
            measurement_rows.append(np.full(feature_shape, np.nan))
            has_data.append(False)
        else:
            measurement_rows.append(evoked_to_feature_vector(evoked))
            has_data.append(True)

    measurements = np.stack(measurement_rows)
    manifest = pd.DataFrame([trial_records[uid] for uid in sorted_uids]).copy()
    manifest.insert(0, "overall_trial_N", np.arange(len(manifest)))
    manifest.insert(1, "condition", condition)
    manifest.insert(2, "level", "trial_lvl")
    manifest["has_data"] = has_data

    obs_descriptors = {
        "trial_uid": manifest["trial_uid"].tolist(),
        "overall_trial_N": manifest["overall_trial_N"].tolist(),
        "sess_N": manifest["sess_N"].tolist(),
        "trial_N": manifest["trial_N"].tolist(),
        "item_id": manifest["item_id"].tolist(),
        "pattern": manifest["pattern"].tolist(),
        "has_data": manifest["has_data"].tolist(),
    }
    dataset = Dataset(
        measurements=measurements,
        descriptors=dataset_descriptors(subj_N, condition, "trial_lvl"),
        obs_descriptors=obs_descriptors,
    )
    return dataset, measurements, manifest


def build_pattern_dataset(
    subj_N: int,
    condition: str,
    trial_measurements: np.ndarray,
    trial_manifest: pd.DataFrame,
    feature_shape: tuple[int, ...],
) -> tuple[Dataset, np.ndarray, pd.DataFrame]:
    pattern_rows = []
    manifest_rows = []
    for pattern in sorted(trial_manifest["pattern"].dropna().unique()):
        pattern_mask = trial_manifest["pattern"].to_numpy() == pattern
        valid_mask = pattern_mask & trial_manifest["has_data"].to_numpy(dtype=bool)
        n_valid = int(valid_mask.sum())
        n_total = int(pattern_mask.sum())
        if n_valid == 0:
            pattern_rows.append(np.full(feature_shape, np.nan))
            has_data = False
        else:
            pattern_rows.append(np.nanmean(trial_measurements[valid_mask], axis=0))
            has_data = True
        manifest_rows.append(
            {
                "condition": condition,
                "level": "pattern_lvl",
                "pattern": pattern,
                "has_data": has_data,
                "n_valid_items": n_valid,
                "n_total_items": n_total,
            }
        )

    measurements = np.stack(pattern_rows)
    manifest = pd.DataFrame(manifest_rows)
    obs_descriptors = {
        "pattern": manifest["pattern"].tolist(),
        "has_data": manifest["has_data"].tolist(),
        "n_valid_items": manifest["n_valid_items"].tolist(),
        "n_total_items": manifest["n_total_items"].tolist(),
    }
    dataset = Dataset(
        measurements=measurements,
        descriptors=dataset_descriptors(subj_N, condition, "pattern_lvl"),
        obs_descriptors=obs_descriptors,
    )
    return dataset, measurements, manifest


def dataset_without_nan_observations(dataset: Dataset) -> tuple[Dataset, np.ndarray]:
    """Return a copy of a Dataset restricted to observations with finite measurements."""
    measurements = np.asarray(dataset.measurements)
    valid_mask = np.isfinite(measurements.reshape(measurements.shape[0], -1)).all(axis=1)
    obs_descriptors = {
        key: np.asarray(value)[valid_mask].tolist()
        for key, value in dataset.obs_descriptors.items()
    }
    valid_dataset = Dataset(
        measurements=measurements[valid_mask],
        descriptors={**dataset.descriptors, "nan_observations_dropped_for_rdm": True},
        obs_descriptors=obs_descriptors,
        channel_descriptors=dataset.channel_descriptors,
    )
    return valid_dataset, valid_mask


def calculate_rdm(dataset: Dataset) -> tuple[RDMs, np.ndarray]:
    """Calculate an RDM and expand it back to the dataset observation grid."""
    valid_dataset, valid_mask = dataset_without_nan_observations(dataset)
    n_observations = dataset.measurements.shape[0]
    full_matrix = np.full((n_observations, n_observations), np.nan, dtype=float)
    valid_indices = np.flatnonzero(valid_mask)

    if valid_dataset.measurements.shape[0] == 1:
        full_matrix[valid_indices[0], valid_indices[0]] = 0.0
    elif valid_dataset.measurements.shape[0] >= 2:
        valid_rdm = calc_rdm(valid_dataset, method=DISSIMILARITY_METRIC)
        valid_matrix = valid_rdm.get_matrices()[0]
        full_matrix[np.ix_(valid_indices, valid_indices)] = valid_matrix

    descriptors = {
        **dataset.descriptors,
        "nan_observations_preserved_in_rdm": True,
        "n_nan_observations": int((~valid_mask).sum()),
    }
    rdm_descriptors = {
        "subj_N": [dataset.descriptors.get("subj_N")],
        "condition": [dataset.descriptors.get("condition")],
        "level": [dataset.descriptors.get("level")],
    }
    full_rdm = RDMs(
        dissimilarities=full_matrix[None, :, :],
        dissimilarity_measure=DISSIMILARITY_METRIC,
        descriptors=descriptors,
        rdm_descriptors=rdm_descriptors,
        pattern_descriptors=dataset.obs_descriptors,
    )
    return full_rdm, valid_mask


def save_level_outputs(
    subj_label: str,
    condition: str,
    level: str,
    dataset: Dataset,
    measurements: np.ndarray,
    manifest: pd.DataFrame,
) -> dict:
    rdm, rdm_valid_mask = calculate_rdm(dataset)
    base = f"{subj_label}-{level}-{condition}-frontal-no_baseline"

    np.save(RUN_DIRS["trial_data"] / f"measurements-{base}.npy", measurements)
    manifest.to_csv(RUN_DIRS["metadata"] / f"manifest-{base}.csv", index=False)
    dataset.save(RUN_DIRS["datasets"] / f"dataset-{base}.hdf5", file_type="hdf5", overwrite=True)
    np.save(RUN_DIRS["metadata"] / f"rdm_valid_mask-{base}.npy", rdm_valid_mask)

    rdm_path = RUN_DIRS["rdms"] / f"rdm-{base}.hdf5"
    rdm_npy_path = RUN_DIRS["rdms"] / f"rdm-{base}.npy"
    rdm.save(rdm_path, file_type="hdf5", overwrite=True)
    rdm_matrix = rdm.get_matrices()
    np.save(rdm_npy_path, rdm_matrix)

    return {
        "level": level,
        "n_dataset_observations": measurements.shape[0],
        "n_valid_for_condition": int(np.sum(manifest["has_data"])),
        "n_nan_observations": int((~manifest["has_data"]).sum()),
        "n_rdm_observations": int(np.sum(rdm_valid_mask)),
        "n_nan_rdm_cells": int(np.isnan(rdm_matrix).sum()),
        "measurements_shape": tuple(measurements.shape),
        "rdm_shape": tuple(rdm_matrix.shape),
        "dataset": str(RUN_DIRS["datasets"] / f"dataset-{base}.hdf5"),
        "rdm": str(rdm_path),
        "rdm_numpy": str(rdm_npy_path),
    }


def save_subject_outputs(
    subj_N: int,
    condition_data: dict[str, dict[str, mne.Evoked]],
    trial_records: dict[str, dict],
    random_window_rows: list[dict],
    aligned_uids: list[str],
) -> dict[str, dict]:
    subj_label = f"subj_{subj_N:02}"
    output_summary: dict[str, dict] = {}
    feature_shape = infer_feature_shape(condition_data)
    sorted_uids = sorted_aligned_uids(aligned_uids, trial_records)

    if EXPORT_EVOKED_OBJECTS:
        save_pickle(
            {
                "condition_data": condition_data,
                "trial_records": trial_records,
                "aligned_trial_uids": sorted_uids,
                "missing_condition_data": "stored as np.nan in datasets/measurements",
            },
            RUN_DIRS["trial_data"] / f"{subj_label}-condition_evokeds.pkl",
        )

    pd.DataFrame(random_window_rows).to_csv(
        RUN_DIRS["trial_data"] / f"{subj_label}-frp_control_random_windows.csv",
        index=False,
    )

    for condition in CONDITIONS:
        trial_dataset, trial_measurements, trial_manifest = build_trial_dataset(
            subj_N=subj_N,
            condition=condition,
            condition_trials=condition_data[condition],
            trial_records=trial_records,
            aligned_uids=sorted_uids,
            feature_shape=feature_shape,
        )
        output_summary[f"trial_lvl/{condition}"] = save_level_outputs(
            subj_label,
            condition,
            "trial_lvl",
            trial_dataset,
            trial_measurements,
            trial_manifest,
        )

        pattern_dataset, pattern_measurements, pattern_manifest = build_pattern_dataset(
            subj_N=subj_N,
            condition=condition,
            trial_measurements=trial_measurements,
            trial_manifest=trial_manifest,
            feature_shape=feature_shape,
        )
        output_summary[f"pattern_lvl/{condition}"] = save_level_outputs(
            subj_label,
            condition,
            "pattern_lvl",
            pattern_dataset,
            pattern_measurements,
            pattern_manifest,
        )

    pd.DataFrame([trial_records[uid] for uid in sorted_uids]).to_csv(
        RUN_DIRS["metadata"] / f"{subj_label}-aligned_trials.csv",
        index=False,
    )
    return output_summary


## Run Experiment 1

This cell may take a while for all participants. For a quick smoke test, set `SUBJ_NS = [1]` in the configuration cell before running.

In [22]:
rng = np.random.default_rng(RNG_SEED)

group = HumanGroupData(
    data_dir=HUMAN_DATA_DIR,
    preprocessed_dir=PREPROCESSED_DIR,
    export_dir=RUN_DIRS["run"],
    subj_Ns=SUBJ_NS,
    data_fmt=DATA_FMT,
)

experiment_summary = []
processing_errors = []

for subj_N, subj in tqdm(group.subjects.items(), desc="Experiment 1 participants"):
    try:
        condition_data, trial_records, random_window_rows = process_subject_trials(subj, rng)
        aligned_uids = dataset_trial_uids(trial_records)
        complete_uids = complete_trial_uids(condition_data)

        status_row = {
            "subj_N": subj_N,
            "n_trials_total": len(trial_records),
            "n_frp": len(condition_data["frp"]),
            "n_frp_control": len(condition_data["frp_control"]),
            "n_response": len(condition_data["response"]),
            "n_rest": len(condition_data["rest"]),
            "n_complete": len(complete_uids),
            "n_dataset_trials": len(aligned_uids),
            "status": "ok",
        }

        if len(aligned_uids) == 0:
            status_row["status"] = "no_trials"
            experiment_summary.append(status_row)
            continue

        output_summary = save_subject_outputs(
            subj_N=subj_N,
            condition_data=condition_data,
            trial_records=trial_records,
            random_window_rows=random_window_rows,
            aligned_uids=aligned_uids,
        )
        status_row["outputs"] = json.dumps(output_summary)
        experiment_summary.append(status_row)

    except Exception as exc:
        processing_errors.append({"subj_N": subj_N, "error": repr(exc)})
        experiment_summary.append(
            {
                "subj_N": subj_N,
                "n_trials_total": np.nan,
                "n_frp": np.nan,
                "n_frp_control": np.nan,
                "n_response": np.nan,
                "n_rest": np.nan,
                "n_complete": np.nan,
                "n_dataset_trials": np.nan,
                "status": "error",
                "error": repr(exc),
            }
        )

summary_df = pd.DataFrame(experiment_summary)
summary_df.to_csv(RUN_DIRS["logs"] / "experiment1_summary.csv", index=False)
pd.DataFrame(processing_errors).to_csv(RUN_DIRS["logs"] / "experiment1_errors.csv", index=False)

summary_df


Experiment 1 participants:   0%|          | 0/1 [00:00<?, ?it/s]

subj 02 sessions:   0%|          | 0/5 [00:00<?, ?it/s]

Removing miscellaneous channels: ['timestamp']
Unique events found: 


/Users/chris/Documents/PhD-Local/abstract_reasoning/analysis/src/ar_analysis/data_loader/human/session.py:309: RuntimeWarning: Omitted 35 annotation(s) that were outside data range.
  raw.set_annotations(annotations)
/Users/chris/Documents/PhD-Local/abstract_reasoning/analysis/src/ar_analysis/data_loader/human/session.py:309: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annotations)


Removing miscellaneous channels: ['timestamp']
Unique events found: 


...

Removing miscellaneous channels: ['timestamp']
Unique events found: 


/Users/chris/Documents/PhD-Local/abstract_reasoning/analysis/src/ar_analysis/data_loader/human/session.py:309: RuntimeWarning: Omitted 6 annotation(s) that were outside data range.
  raw.set_annotations(annotations)


Removing miscellaneous channels: ['timestamp']
Unique events found: 


/Users/chris/Documents/PhD-Local/abstract_reasoning/analysis/src/ar_analysis/data_loader/human/session.py:309: RuntimeWarning: Omitted 9 annotation(s) that were outside data range.
  raw.set_annotations(annotations)


,subj_N,n_trials_total,n_frp,n_frp_control,n_response,n_rest,n_complete,n_dataset_trials,status,outputs
0,2,400,358,225,400,400,225,400,ok,"{""trial_lvl/frp"": {""level"": ""trial_lvl"", ""n_da..."


## Sanity Checks

Use these after the run to verify that all conditions have matched trials and equal feature dimensionality per participant.

In [23]:
summary_df = pd.read_csv(RUN_DIRS["logs"] / "experiment1_summary.csv")
summary_df

,subj_N,n_trials_total,n_frp,n_frp_control,n_response,n_rest,n_complete,n_dataset_trials,status,outputs
0,2,400,358,225,400,400,225,400,ok,"{""trial_lvl/frp"": {""level"": ""trial_lvl"", ""n_da..."


In [26]:
def inspect_saved_subject(subj_N: int):
    subj_label = f"subj_{subj_N:02}"
    manifests = sorted(RUN_DIRS["metadata"].glob(f"manifest-{subj_label}-*.csv"))
    measurements = sorted(RUN_DIRS["trial_data"].glob(f"measurements-{subj_label}-*.npy"))
    print("Manifests:")
    for fpath in manifests:
        df = pd.read_csv(fpath)
        if "trial_uid" in df.columns:
            obs_label = f"trials={df['trial_uid'].nunique()}"
        elif "pattern" in df.columns:
            obs_label = f"patterns={df['pattern'].nunique()}"
        else:
            obs_label = f"observations={len(df)}"
        level = df["level"].iloc[0] if "level" in df.columns and len(df) else "unknown"
        print(f"  {fpath.name}: {df.shape}, level={level}, {obs_label}")
    print("Measurements:")
    for fpath in measurements:
        arr = np.load(fpath)
        print(f"  {fpath.name}: {arr.shape}")


ok_subjects = summary_df.query("status == 'ok'")["subj_N"].tolist()
if ok_subjects:
    inspect_saved_subject(int(ok_subjects[0]))
else:
    print("No completed subjects to inspect.")


Manifests:
  manifest-subj_02-pattern_lvl-frp-frontal-no_baseline.csv: (8, 6), level=pattern_lvl, patterns=8
  manifest-subj_02-pattern_lvl-frp_control-frontal-no_baseline.csv: (8, 6), level=pattern_lvl, patterns=8
  manifest-subj_02-pattern_lvl-response-frontal-no_baseline.csv: (8, 6), level=pattern_lvl, patterns=8
  manifest-subj_02-pattern_lvl-rest-frontal-no_baseline.csv: (8, 6), level=pattern_lvl, patterns=8
  manifest-subj_02-trial_lvl-frp-frontal-no_baseline.csv: (400, 11), level=trial_lvl, trials=400
  manifest-subj_02-trial_lvl-frp_control-frontal-no_baseline.csv: (400, 11), level=trial_lvl, trials=400
  manifest-subj_02-trial_lvl-response-frontal-no_baseline.csv: (400, 11), level=trial_lvl, trials=400
  manifest-subj_02-trial_lvl-rest-frontal-no_baseline.csv: (400, 11), level=trial_lvl, trials=400
Measurements:
  measurements-subj_02-pattern_lvl-frp-frontal-no_baseline.npy: (8, 20910)
  measurements-subj_02-pattern_lvl-frp_control-frontal-no_baseline.npy: (8, 20910)
  measure

## Notes for Later Extensions

- `trial_lvl` and `pattern_lvl` datasets/RDMs are exported explicitly. Do not use descriptor-level RDM export from the trial dataset for this pipeline.
- Current random controls sample windows from `RANDOM_CONTROL_START_EVENT` to `RANDOM_CONTROL_END_EVENT` with random onsets separated by at least `RANDOM_CONTROL_MIN_ONSET_DIFF_S`. For an alternative control window, change `RANDOM_CONTROL_START_EVENT` and/or `RANDOM_CONTROL_END_EVENT`.
- Current trial representations flatten frontal channels x time. If we want temporal RDMs, switch to `rsatoolbox.data.TemporalDataset` instead of flattening.